# ADVENTINST UNIVERSITY OF CENTRAL AFRICA (AUCA)
### Week 12 Lab — Big Data Tools: SQL · APIs · PySpark

<br>
<br>

**Student Name:** Mutangana Joseph

**Student ID:** 29061

**Course Name:** Intorduction to Big Data Analytics

**Program:** Software Engineering

**Faculty:** Information Technology


# Part 1 · SQL with Python (12 marks)

## Loads the CSV into an SQLite database

In [1]:

import pandas as pd
import sqlite3

conn = sqlite3.connect("momo.db")
df = pd.read_csv("week12_transactions.csv")
df.to_sql("transactions", conn, if_exists="replace", index=False)

2000

## Exercise 1.1 — SELECT / WHERE / ORDER BY
**a) transactions above 100,000 RWF**

In [2]:
query="""
select txn_id, district, amount_rwf
from transactions
where amount_rwf >100000 and txn_type='send'
order by amount_rwf desc limit 5;
"""

results = pd.read_sql(query, conn)
results

,txn_id,district,amount_rwf
0,T01708,Kicukiro,140000
1,T00364,Huye,139000
2,T00895,Muhanga,139000
3,T00899,Nyarugenge,139000
4,T01314,Gasabo,139000


**b) Count flagged transactions**

In [3]:
query="""
select count(*) as flagged_transactions
from transactions 
where flagged = 1
"""

results = pd.read_sql(query, conn)
results

,flagged_transactions
0,11


## Exercise 1.2 — GROUP BY

**a) which type moves the most money?**

In [4]:
query = """
select count(*) as count, sum(amount_rwf) as total_amount, 
avg(amount_rwf) as amount_average, txn_type
from transactions
group by txn_type order by sum(amount_rwf) desc
"""
results = pd.read_sql(query, conn)
results

,count,total_amount,amount_average,txn_type
0,868,62049000,71485.023041,send
1,330,29970000,90818.181818,paybill
2,489,27024000,55263.803681,receive
3,313,833000,2661.341853,airtime


**b)  which district leads?**

In [5]:
query = """
select count(*) as count, sum(amount_rwf) as total_amount, 
avg(amount_rwf) as amount_average, district
from transactions
group by district order by sum(amount_rwf) desc
"""
results = pd.read_sql(query, conn)
results

,count,total_amount,amount_average,district
0,247,21076000,85327.935223,Muhanga
1,255,20738000,81325.490196,Gasabo
2,251,18736000,74645.418327,Huye
3,252,18091000,71789.682540,Nyarugenge
4,249,11338000,45534.136546,Kicukiro
5,253,10490500,41464.426877,Musanze
6,246,10110500,41099.593496,Rubavu
7,247,9296000,37635.627530,Nyagatare


## Exercise 1.3 — The busiest hour

**(a) Transactions per hour**

In [6]:
query="""
select hour, count(*) as transactions
from transactions
group by hour order by transactions desc limit 5
"""

results = pd.read_sql(query, conn)
results

,hour,transactions
0,20,185
1,17,183
2,18,176
3,12,170
4,11,169


**(b) Transactions per hour using pandas**

In [7]:
busiest_hour = df.groupby("hour").size().sort_values(ascending=False)
print(busiest_hour)

hour
20    185
17    183
18    176
12    170
11    169
10    168
8     168
19    158
9     144
16     38
3      37
13     35
15     35
14     34
2      34
4      34
0      33
6      33
5      33
1      32
21     26
7      25
22     25
23     25
dtype: int64


## Part 2 · APIs & JSON (10 marks)

## Exercise 2.1 — JSON by hand

**a) Parse the json string with the json.loads**

In [8]:
import json, requests

json_string="""{
    "city": "kigali",
    "readings": [30,12,25,24]}"""
data = json.loads(json_string)


**a) Print the city**

In [9]:
print(data["city"])

kigali


**b) Print the number of readings**

In [10]:
print(f"Number of readings: {len(data["readings"])}")

Number of readings: 4


**c) Print the average of readings**

In [11]:
print(f"Average of readings: {round(sum(data["readings"])/len(data["readings"]),2)}°C")

Average of readings: 22.75°C


## Exercise 2.2 — Call a real API

In [12]:

try:
    url = ("https://api.open-meteo.com/v1/forecast"
           "?latitude=-1.9441&longitude=30.0619"
           "&hourly=temperature_2m&forecast_days=1&timezone=Africa%2FKigali")
    data = requests.get(url, timeout=10).json()
except Exception:
    with open("week12_api_sample.json") as f:
        data = json.load(f)

times = data["hourly"]["time"]
temps = data["hourly"]["temperature_2m"]

**a) How many hours**

In [13]:
print(len(data["hourly"]["time"]))

24


**b) Min and Max temperature**

In [14]:
print(f"Minimum temperature: {min(data['hourly']['temperature_2m'])}°C")
print(f"Maximum temperature: {max(data['hourly']['temperature_2m'])}°C")

Minimum temperature: 16.3°C
Maximum temperature: 24.5°C


**c) Time of the hotest day**

In [15]:
max_temp=max(data['hourly']['temperature_2m'])
pos=data['hourly']['temperature_2m'].index(max_temp)
print(f"Time of the hotest day: {data['hourly']['time'][pos]}")

Time of the hotest day: 2026-11-22T15:00


## Exercise 2.3 — API data meets your toolbox

**Create new  DataFrame**

In [16]:
new_df = pd.DataFrame({"time": times, "temp_c": temps})

**a) Describe the new  DataFrame**

In [17]:
new_df.describe()

,temp_c
count,24.000000
mean,19.708333
std,2.826108
min,16.300000
25%,17.025000
50%,19.250000
75%,21.750000
max,24.500000


**b) Count hot hours**

In [18]:
temp_c=new_df["temp_c"]
mask=temp_c >=25
print(f"Number of hot hot hours: {mask.sum()}")

Number of hot hot hours: 0


# Part 3 · PySpark

## Exercise 3.1 — Load & inspect

In [25]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("momo_transactions").master("local[*]").getOrCreate()
sdf=spark.read.csv("week12_transactions.csv", header=True, inferSchema=True)

**(a) Count transactions above 100,000 RWF**

In [20]:
filtered_df = sdf.filter(sdf.amount_rwf > 100000).count()
print(f"Transactions above 100k: {filtered_df}")

Transactions above 100k: 451


**b) Show 5 flagged transactions**

In [21]:
sdf.filter(sdf.flagged == 1).show()

+------+----------+----+----------+--------+----------+-------+
|txn_id|      date|hour|  district|txn_type|amount_rwf|flagged|
+------+----------+----+----------+--------+----------+-------+
|T00279|2026-11-29|  19|   Muhanga|    send|    127000|      1|
|T00326|2026-11-07|  22|      Huye|    send|    127000|      1|
|T00536|2026-11-27|  17|    Gasabo|    send|    127000|      1|
|T00655|2026-11-30|  17|    Rubavu|    send|    134000|      1|
|T00697|2026-11-01|   9|   Muhanga|    send|    127000|      1|
|T00864|2026-11-29|  17|    Gasabo|    send|    127000|      1|
|T00948|2026-11-15|  17|    Gasabo|    send|    127000|      1|
|T01008|2026-11-12|  20| Nyagatare|    send|    134000|      1|
|T01033|2026-11-15|   2|Nyarugenge|    send|    127000|      1|
|T01368|2026-11-15|   9|      Huye|    send|    127000|      1|
|T01552|2026-11-18|  17|   Musanze|    send|    134000|      1|
+------+----------+----+----------+--------+----------+-------+



## Exercise 3.3 — groupBy, fourth reunion

**(a) Total amount per txn_type with**

In [22]:
sdf.groupBy(sdf.txn_type).sum("amount_rwf").orderBy("sum(amount_rwf)", ascending=False).show()

+--------+---------------+
|txn_type|sum(amount_rwf)|
+--------+---------------+
|    send|       62049000|
| paybill|       29970000|
| receive|       27024000|
| airtime|         833000|
+--------+---------------+



**c) Do my spark Answer matches SQL answers?**

## YES! They match.

# Bonus (+3) — Spark speaks SQL too

**Group by TXN_TYPE**

In [23]:
sdf.createOrReplaceTempView("transactions")
query="""select count(*) as count, sum(amount_rwf) as total_amount, 
avg(amount_rwf) as amount_average, txn_type
from transactions
group by txn_type order by sum(amount_rwf) desc"""

spark.sql(query).show()

+-----+------------+------------------+--------+
|count|total_amount|    amount_average|txn_type|
+-----+------------+------------------+--------+
|  868|    62049000| 71485.02304147465|    send|
|  330|    29970000| 90818.18181818182| paybill|
|  489|    27024000|  55263.8036809816| receive|
|  313|      833000|2661.3418530351437| airtime|
+-----+------------+------------------+--------+



**GroupBy District**

In [24]:
sdf.createOrReplaceTempView("transactions")
query="""select count(*) as count, sum(amount_rwf) as total_amount, 
avg(amount_rwf) as amount_average, district
from transactions
group by district order by sum(amount_rwf) desc"""

spark.sql(query).show()

+-----+------------+-----------------+----------+
|count|total_amount|   amount_average|  district|
+-----+------------+-----------------+----------+
|  247|    21076000|85327.93522267207|   Muhanga|
|  255|    20738000|81325.49019607843|    Gasabo|
|  251|    18736000|74645.41832669322|      Huye|
|  252|    18091000|71789.68253968254|Nyarugenge|
|  249|    11338000|45534.13654618474|  Kicukiro|
|  253|    10490500|41464.42687747035|   Musanze|
|  246|    10110500|41099.59349593496|    Rubavu|
|  247|     9296000|37635.62753036437| Nyagatare|
+-----+------------+-----------------+----------+



## Why is this powerful?
You can write familiar SQL queries against Spark DataFrames, even when Spark is processing very large datasets across multiple machines.


# Part 4 — Choose the Right Tool

| Scenario | Best Tool | Reason |
|---|---|---|
| Loading and exploring a small CSV dataset | **Pandas** | It makes it easy to load, explore and manipulate data using Python. |
| Filtering, joining and aggregating structured records | **SQL** | SQL is designed for querying and managing structured data. |
| Processing billions of records across multiple machines | **Spark** | Spark supports distributed processing across multiple machines. |
| Retrieving live weather or external data | **API** | APIs allow applications to communicate with external services and retrieve data. |
